# 강의 04 · 실습 2 — 체크포인트와 사람 개입 · (1) 강사 시연

## 1. 문제상황

- 온라인 쇼핑몰 고객센터는 고객 한 사람과 여러 통의 메일을 주고받습니다.
- 답장을 쓰려면 그 고객과 앞서 주고받은 메일을 다시 읽어야 하고, 메일이 쌓일수록 다시 읽는 양이 늘어납니다.
- 답장은 발송 전에 담당자가 확인해야 하는데, 프로그램이 답장을 만들자마자 보내 버리면 담당자가 확인할 자리가 없습니다.
- 처리 도중 프로그램이 꺼지면, 이미 만들어 둔 답장 초안과 확인 결과가 사라져서 처음부터 다시 만들어야 합니다.

## 2. 문제와 목표

- **문제**: 앞서 주고받은 메일을 사람이 다시 읽어야 하고, 발송 전에 사람이 확인할 자리가 없고, 프로그램이 꺼지면 진행 상황이 사라집니다.
- **목표**
  - 고객 한 사람의 메일을 같은 `thread_id` 아래 이어서 받습니다.
  - 답장 초안을 만든 뒤 발송 앞에서 멈춰 담당자의 승인 또는 수정을 받습니다.
  - 노드가 끝날 때마다 진행 상황을 파일에 저장해, 프로그램이 꺼져도 마지막 저장 지점부터 이어 가는 처리 흐름을 만듭니다.
  - 대화가 길어지면 오래된 메시지를 요약으로 바꿔 모델에 넣는 대화 기록의 길이를 일정하게 유지합니다.
- **목표 달성 여부의 판정 기준**: 같은 `thread_id`로 고객 메일 세 통을 차례로 넣었을 때,
  - 메일마다 그래프가 human_review 노드에서 멈추고 담당자의 답을 받은 뒤에만 발송 줄이 출력되고,
  - 두 번째 메일의 발송 단계에서 프로그램이 실패한 뒤, 새로 만든 그래프 객체가 같은 `thread_id`의 저장 지점(다음 노드 = send)을 읽어 초안을 다시 만들지 않고 발송만 이어 가며,
  - 세 번째 메일에서 대화 기록의 앞부분이 요약 메시지로 바뀌어 있는 것을 실행 결과에서 확인합니다.
  - 담당자의 방침은 대본으로 미리 넣습니다.

## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec04_ex02_s1_diagram.svg)

## 4. 단계별 요구사항

1. **상태를 정의합니다.**
    - 대화 기록(`messages`), 답장 초안(`draft`), 담당자 결정(`decision`), 발송 여부(`sent`) 키 네 개를 가지는 상태를 선언합니다.
    - `messages` 키에는 `add_messages` 리듀서를 붙여, 노드가 돌려준 메시지가 기존 값을 덮지 않고 뒤에 쌓이게 합니다.
2. **컨텍스트 관리 노드를 만듭니다.**
    - manage 노드는 `messages`의 길이가 `KEEP` 이하이면 아무것도 바꾸지 않습니다.
    - `KEEP`을 넘으면 최근 `KEEP` 개만 원문으로 남기고, 그 앞의 메시지들을 모델로 요약한 뒤, 오래된 메시지들을 `RemoveMessage`로 지우고 요약 `SystemMessage` 하나를 넣습니다.
3. **초안 노드를 만듭니다.**
    - draft 노드는 `messages` 전체를 모델에 넣어, 마지막 고객 메일에 답하는 두 문장짜리 답장 초안을 `draft` 키에 씁니다.
4. **검토 노드를 만듭니다.**
    - human_review 노드는 `interrupt()`로 실행을 멈추고 질문과 초안을 담당자에게 보냅니다.
    - 담당자의 답이 문자열이면 그대로 `decision` 키에 쓰고, `edit` 키를 가진 딕셔너리이면 그 값으로 `draft` 키를 교체하고 `decision` 키에 「수정 후 승인」을 씁니다.
5. **발송 노드를 만듭니다.**
    - send 노드는 초안을 발송하고(이 실습에서 발송은 화면 출력으로 대신합니다), 보낸 답장을 `AIMessage`로 `messages` 키에 쌓고, `sent` 키에 `True`를 씁니다.
    - 발송 서버 장애를 모의로 만드는 스위치 `SEND_FAILS`가 켜져 있으면 발송 대신 예외를 일으킵니다.
6. **그래프에 노드를 등록합니다.**
    - 네 노드를 이름과 함께 그래프에 등록합니다.
7. **엣지를 연결합니다.**
    - START → manage → draft → human_review → send → END를 고정 엣지로 연결합니다.
    - 이 실습에는 조건부 엣지가 없습니다.
8. **체크포인터를 장착해 컴파일합니다.**
    - SQLite 파일에 저장하는 체크포인터(`SqliteSaver`)를 열어 `compile(checkpointer=…)`에 넘깁니다.
    - 실행할 때마다 `thread_id`를 담은 설정을 함께 넘깁니다.
9. **그래프를 실행합니다.**
    - 같은 `thread_id`로 고객 메일 세 통을 차례로 넣습니다.
    - 메일마다 멈춘 지점(`next`)과 담당자에게 간 내용을 출력하고 `Command(resume=…)`으로 이어 갑니다.
    - 두 번째 메일에서는 발송을 실패시킨 뒤, 새 체크포인터와 새 그래프 객체로 같은 `thread_id`의 저장 상태를 읽고(`get_state`) 입력 `None`으로 이어 갑니다.
    - 세 번째 메일에서는 대화 기록의 앞부분이 요약으로 바뀐 것을 출력합니다.

## 5. 코드 골격 — LangGraph 5단

랭그래프(LangGraph)로 그래프를 세우는 순서는 다음 다섯 단계입니다. 아래 「6. 코드 — 스텝바이스텝」의 코드 셀이 이 다섯 단계와 하나씩 대응합니다. 체크포인터와 `interrupt()`는 새 단계가 아니라 ⑤ 컴파일과 실행 단계의 확장입니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 상태 정의 | 노드들이 함께 읽고 쓸 키를 선언합니다 | `class ReplyState(TypedDict)`, `Annotated[list, add_messages]` | 1 |
| ② 노드 함수 정의 | 상태를 받아 바뀐 키만 돌려주는 함수를 만듭니다 | `def manage(state) -> dict`, `interrupt()` | 2, 3, 4, 5 |
| ③ 그래프 빌더 생성과 노드 등록 | 빈 그래프를 열고 함수에 이름을 붙여 등록합니다 | `StateGraph(ReplyState)`, `add_node` | 6 |
| ④ 엣지 연결 | 노드 사이의 순서를 정합니다 | `add_edge` | 7 |
| ⑤ 컴파일과 실행 | 체크포인터를 달아 컴파일하고, `thread_id`를 넘겨 실행하고, 멈춘 지점에서 이어 갑니다 | `compile(checkpointer=…)`, `invoke`, `Command(resume=…)`, `get_state`, `invoke(None, …)` | 8, 9 |

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 모델을 준비합니다. 체크포인트를 저장할 파일 위치도 여기서 정합니다.

- API 키는 `.env` 파일에서 읽습니다.
- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- `.env` 파일에는 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

- 체크포인트 파일은 실행할 때마다 새 임시 폴더에 만듭니다. 지난 실행의 저장 기록이 이번 실행에 섞이지 않게 하기 위해서입니다.

In [1]:
import sqlite3
import tempfile
from pathlib import Path
import os

from dotenv import load_dotenv, find_dotenv
from typing import Annotated, TypedDict

from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, HumanMessage, RemoveMessage, SystemMessage
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.types import Command, interrupt

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")

llm = init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")

DB_PATH = Path(tempfile.mkdtemp(prefix="lec04_ex02_")) / "checkpoint.db"   # 체크포인트를 저장할 파일


def show_messages(msgs: list) -> None:
    """대화 기록을 메시지 종류와 앞부분 글자만 한 줄씩 출력한다."""
    for m in msgs:
        print(f"      [{type(m).__name__}] {str(m.content)[:90]}")


print("모델 준비를 마쳤습니다. 체크포인트 파일:", DB_PATH.name)

모델 준비를 마쳤습니다. 체크포인트 파일: checkpoint.db


### 단계 ① — 상태 정의 (요구사항 1)

그래프가 도는 동안 모든 노드가 함께 읽고 쓰는 키를 선언합니다. `messages` 키에 붙인 `add_messages`가 리듀서(reducer)입니다. 노드가 `messages`에 메시지를 돌려주면 리듀서가 기존 목록 뒤에 붙이고, `RemoveMessage`를 돌려주면 같은 `id`의 메시지를 지웁니다.

In [2]:
class ReplyState(TypedDict):
    messages: Annotated[list, add_messages]   # 대화 기록 (고객 메일과 보낸 답장이 쌓인다)
    draft: str      # 답장 초안
    decision: str   # 담당자 결정 (승인 / 수정 후 승인)
    sent: bool      # 발송 여부


print("상태의 키:", list(ReplyState.__annotations__))

상태의 키: ['messages', 'draft', 'decision', 'sent']


### 단계 ② — 노드 함수 정의 (요구사항 2, 3, 4, 5)

- 노드는 상태를 인자로 받아 딕셔너리를 돌려주는 파이썬 함수입니다. 돌려준 딕셔너리가 상태의 해당 키에 반영됩니다.
- `KEEP`과 `SUMMARY_RULE`은 manage 노드가 따르는 원칙입니다. 노드는 판단하지 않고 이 원칙대로 줄입니다.
- `interrupt()`는 노드 실행을 그 자리에서 멈추고, 담당자가 준 값을 그 호출의 반환값으로 받아 이어 가는 함수입니다.
- `SEND_FAILS`는 발송 서버 장애를 모의로 만드는 스위치입니다. 단계 ⑤에서 켜고 끕니다.

In [3]:
KEEP = 3   # 원문으로 남기는 최근 메시지 수
SUMMARY_RULE = ("다음 고객센터 대화를 고객이 문의한 내용과 상담원이 약속한 처리 중심으로 "
                "두 문장 이내 한국어로 요약한다. 새 정보를 지어내지 않는다.")
SEND_FAILS = False   # True이면 send 노드가 발송 서버 장애를 모의로 만들어 예외를 일으킨다


def manage(state: ReplyState) -> dict:
    """대화 기록이 KEEP을 넘으면 오래된 부분을 요약 메시지 하나로 바꾼다."""
    msgs = state["messages"]
    if len(msgs) <= KEEP:
        return {}
    old = msgs[:-KEEP]
    summary = llm.invoke([SystemMessage(SUMMARY_RULE)] + old).content.strip()
    print(f"    [manage] 오래된 메시지 {len(old)}개를 요약으로 바꿉니다")
    return {"messages": [RemoveMessage(id=m.id) for m in old]
            + [SystemMessage(f"[이전 대화 요약] {summary}")]}


def draft(state: ReplyState) -> dict:
    """대화 기록 전체를 보고 마지막 고객 메일에 답하는 초안을 두 문장으로 쓴다."""
    res = llm.invoke([
        SystemMessage("온라인 쇼핑몰 고객센터 상담원이다. 지금까지의 대화를 참고해 "
                      "마지막 고객 메일에 답하는 정중한 답장을 두 문장으로 쓴다."),
        *state["messages"],
    ])
    return {"draft": res.content.strip()}


def human_review(state: ReplyState) -> dict:
    """발송 앞에서 멈추고, 담당자의 답을 받아 결정을 기록한다."""
    answer = interrupt({"질문": "이 초안을 발송할까요?", "초안": state["draft"]})
    if isinstance(answer, dict) and "edit" in answer:
        return {"draft": answer["edit"], "decision": "수정 후 승인"}
    return {"decision": str(answer)}


def send(state: ReplyState) -> dict:
    """초안을 발송하고 (여기서는 화면 출력으로 대신한다) 보낸 답장을 대화 기록에 쌓는다."""
    if SEND_FAILS:
        raise RuntimeError("[강제 예외] 발송 서버에 연결할 수 없습니다")
    print(f"    [발송] ({state['decision']}) {state['draft'][:40]}...")
    return {"messages": [AIMessage(state["draft"])], "sent": True}

### 단계 ③ — 그래프 빌더 생성과 노드 등록 (요구사항 6)

`StateGraph`에 상태를 넘겨 빈 그래프를 열고, `add_node`로 함수마다 이름을 붙여 등록합니다.

In [4]:
g = StateGraph(ReplyState)
g.add_node("manage", manage)
g.add_node("draft", draft)
g.add_node("human_review", human_review)
g.add_node("send", send)

print("등록한 노드:", list(g.nodes))

등록한 노드: ['manage', 'draft', 'human_review', 'send']


### 단계 ④ — 엣지 연결 (요구사항 7)

이 실습의 노드는 한 줄로 이어집니다. `add_edge`로 고정 엣지 다섯 개를 놓습니다. 조건부 엣지는 없습니다.

In [5]:
g.add_edge(START, "manage")
g.add_edge("manage", "draft")
g.add_edge("draft", "human_review")
g.add_edge("human_review", "send")
g.add_edge("send", END)

print("고정 엣지 다섯 개를 놓았습니다.")

고정 엣지 다섯 개를 놓았습니다.


### 단계 ⑤ — 컴파일과 실행 (요구사항 8, 9)

`compile(checkpointer=…)`에 체크포인터를 넘기면 노드가 끝나는 경계마다 상태 전체가 저장됩니다. 저장은 `thread_id` 단위로 나뉘므로, 실행할 때마다 `thread_id`를 담은 설정을 함께 넘깁니다. `SqliteSaver`는 상태를 파일에 두므로, 프로그램을 다시 켜도 같은 파일에서 이어받을 수 있습니다. 아래 다섯 셀(⑤-a ~ ⑤-e)이 모두 이 한 단계에 속합니다.

#### 단계 ⑤-a — 체크포인터 장착과 컴파일

체크포인터를 열어 `compile(checkpointer=…)`에 넘기고, 이 실습 내내 쓸 `thread_id` 설정을 만듭니다.

In [6]:
def open_saver(db_path: Path) -> SqliteSaver:
    """SQLite 파일에 체크포인트를 저장하는 체크포인터를 연다."""
    conn = sqlite3.connect(db_path, check_same_thread=False)
    saver = SqliteSaver(conn)
    saver.setup()
    return saver


graph = g.compile(checkpointer=open_saver(DB_PATH))
THREAD = {"configurable": {"thread_id": "customer-1024"}}
print("체크포인터를 달아 컴파일했습니다. thread_id =", THREAD["configurable"]["thread_id"])

체크포인터를 달아 컴파일했습니다. thread_id = customer-1024


#### 단계 ⑤-b — 실행 1: 멈춤과 승인

첫 메일을 넣으면 그래프가 human_review 노드에서 멈춥니다. `invoke`의 반환값에 `__interrupt__` 키가 들어 있고, `get_state`의 `next`가 멈춘 지점입니다. `Command(resume="승인")`으로 이어 가면 발송까지 진행됩니다.

In [7]:
EMAILS = [
    "결제가 두 번 청구된 것 같습니다. 확인 부탁드립니다.",
    "아까 문의한 건, 환불은 언제쯤 들어오나요?",
    "환불 확인했습니다. 그런데 같이 주문한 다른 상품은 아직 배송 전인가요?",
]

print("=== 실행 1: 1번 메일 ===")
out = graph.invoke({"messages": [HumanMessage(EMAILS[0])], "sent": False}, THREAD)
print("  [멈춤 여부] __interrupt__ 있음 =", "__interrupt__" in out)
print("  [멈춘 지점] next =", graph.get_state(THREAD).next)
print("  [담당자에게 간 내용]", out["__interrupt__"][0].value)
final = graph.invoke(Command(resume="승인"), THREAD)
print(f"  [재개 후] decision={final['decision']!r} sent={final['sent']}")

=== 실행 1: 1번 메일 ===


  [멈춤 여부] __interrupt__ 있음 = True
  [멈춘 지점] next = ('human_review',)
  [담당자에게 간 내용] {'질문': '이 초안을 발송할까요?', '초안': '결제가 이중으로 청구된 점 확인을 위해 주문번호와 결제일시를 알려주시면 즉시 확인하겠습니다. 중복 결제가 확인될 경우 한 건은 신속히 취소 또는 환불 처리해 드리겠습니다.'}
    [발송] (승인) 결제가 이중으로 청구된 점 확인을 위해 주문번호와 결제일시를 알려주시면 ...
  [재개 후] decision='승인' sent=True


#### 단계 ⑤-c — 실행 2: 수정 후 발송 실패

같은 `thread_id`로 두 번째 메일을 넣습니다. 담당자가 수정문을 주고, 발송 서버 장애 스위치를 켠 채 이어 갑니다. send 노드가 예외를 일으켜 프로그램이 멈추지만, human_review까지의 결과는 이미 파일에 저장되어 있습니다.

In [8]:
print("=== 실행 2: 2번 메일 (같은 thread_id) ===")
out = graph.invoke({"messages": [HumanMessage(EMAILS[1])], "sent": False}, THREAD)
print("  [멈춘 지점] next =", graph.get_state(THREAD).next)
print("  [모델이 쓴 초안]", out["draft"])

EDIT = "고객님, 이중 청구 건은 확인되어 오늘 환불 처리했습니다. 영업일 기준 3일 안에 입금됩니다."
SEND_FAILS = True   # 발송 서버 장애를 모의로 만든다
try:
    graph.invoke(Command(resume={"edit": EDIT}), THREAD)
except RuntimeError as e:
    print("  [예외]", e)
snap = graph.get_state(THREAD)
print("  [저장된 상태] decision =", snap.values["decision"])
print("  [저장된 상태] next =", snap.next)

=== 실행 2: 2번 메일 (같은 thread_id) ===


  [멈춘 지점] next = ('human_review',)
  [모델이 쓴 초안] 환불 처리가 완료되면 결제수단에 따라 영업일 기준 3~7일 이내에 반영됩니다. 아직 환불 처리가 진행되지 않았다면 주문번호를 알려주시면 현재 상태를 확인해 안내해 드리겠습니다.
  [예외] [강제 예외] 발송 서버에 연결할 수 없습니다
  [저장된 상태] decision = 수정 후 승인
  [저장된 상태] next = ('send',)


#### 단계 ⑤-d — 다시 켠 프로그램에서 이어 가기

새 연결로 체크포인터를 다시 열고 새 그래프 객체를 만듭니다. 다시 켠 프로그램에 해당합니다. `get_state`는 어떤 노드도 실행하지 않고 저장된 상태를 읽습니다. 입력 위치의 `None`이 「새로 시작이 아니라 이어서」의 표시입니다.

In [9]:
graph2 = g.compile(checkpointer=open_saver(DB_PATH))   # 새 연결·새 그래프 객체 = 다시 켠 프로그램
snap = graph2.get_state(THREAD)
print("=== 다시 켠 프로그램: 저장 파일에서 이어받은 상태 ===")
print("  [이어받은 상태] next =", snap.next)
print("  [이어받은 상태] draft =", snap.values["draft"])

SEND_FAILS = False   # 발송 서버가 복구되었다
final = graph2.invoke(None, THREAD)   # None = 새 입력 없이 저장 지점부터 이어 간다
print(f"  [재개 후] decision={final['decision']!r} sent={final['sent']}")
print("  [대화 기록]")
show_messages(final["messages"])

=== 다시 켠 프로그램: 저장 파일에서 이어받은 상태 ===
  [이어받은 상태] next = ('send',)
  [이어받은 상태] draft = 고객님, 이중 청구 건은 확인되어 오늘 환불 처리했습니다. 영업일 기준 3일 안에 입금됩니다.
    [발송] (수정 후 승인) 고객님, 이중 청구 건은 확인되어 오늘 환불 처리했습니다. 영업일 기준 ...
  [재개 후] decision='수정 후 승인' sent=True
  [대화 기록]
      [HumanMessage] 결제가 두 번 청구된 것 같습니다. 확인 부탁드립니다.
      [AIMessage] 결제가 이중으로 청구된 점 확인을 위해 주문번호와 결제일시를 알려주시면 즉시 확인하겠습니다. 중복 결제가 확인될 경우 한 건은 신속히 취소 또는 환불 처리해 드리
      [HumanMessage] 아까 문의한 건, 환불은 언제쯤 들어오나요?
      [AIMessage] 고객님, 이중 청구 건은 확인되어 오늘 환불 처리했습니다. 영업일 기준 3일 안에 입금됩니다.


#### 단계 ⑤-e — 실행 3: 컨텍스트 관리

대화 기록이 `KEEP`을 넘은 상태에서 세 번째 메일을 넣습니다. manage 노드가 오래된 메시지들을 요약 메시지 하나로 바꾼 뒤 draft 노드가 초안을 씁니다.

In [10]:
print("=== 실행 3: 3번 메일 (같은 thread_id) ===")
out = graph2.invoke({"messages": [HumanMessage(EMAILS[2])], "sent": False}, THREAD)
print("  [manage 뒤의 대화 기록]")
show_messages(out["messages"])
print("  [모델이 쓴 초안]", out["draft"])
final = graph2.invoke(Command(resume="승인"), THREAD)
print(f"  [재개 후] decision={final['decision']!r} sent={final['sent']}")
print("  [최종 대화 기록 길이]", len(final["messages"]))

=== 실행 3: 3번 메일 (같은 thread_id) ===


    [manage] 오래된 메시지 2개를 요약으로 바꿉니다


  [manage 뒤의 대화 기록]
      [HumanMessage] 아까 문의한 건, 환불은 언제쯤 들어오나요?
      [AIMessage] 고객님, 이중 청구 건은 확인되어 오늘 환불 처리했습니다. 영업일 기준 3일 안에 입금됩니다.
      [HumanMessage] 환불 확인했습니다. 그런데 같이 주문한 다른 상품은 아직 배송 전인가요?
      [SystemMessage] [이전 대화 요약] 고객은 결제가 두 번 청구된 것 같아 확인을 요청했습니다. 상담원은 주문번호와 결제일시를 받으면 확인하고, 중복 결제 확인 시 한 건을 취소 
  [모델이 쓴 초안] 함께 주문하신 상품의 배송 상태는 주문번호 확인이 필요하오니, 주문번호를 알려주시면 바로 확인해 드리겠습니다. 확인 후 아직 출고 전이라면 출고 예정일과 배송 일정을 함께 안내해 드리겠습니다.
    [발송] (승인) 함께 주문하신 상품의 배송 상태는 주문번호 확인이 필요하오니, 주문번호를...
  [재개 후] decision='승인' sent=True
  [최종 대화 기록 길이] 5


## 7. 실행 결과 확인

위 실행 결과에서 다음 네 가지를 확인합니다.

1. 실행 1에서 `__interrupt__ 있음 = True`, `next = ('human_review',)`가 출력되고, 담당자에게 간 내용에 질문과 초안이 들어 있습니다. `[발송]` 줄은 `Command(resume="승인")` 뒤에만 출력됩니다.
2. 실행 2는 새 그래프를 만들지 않고 같은 `thread_id`로 두 번째 메일만 넣었는데, 뒤의 `[대화 기록]`에 1번 메일과 그 답장이 그대로 남아 있습니다. 대화 기록이 `thread_id` 아래 저장되어 있기 때문입니다. 예외 뒤에 저장된 상태는 `decision = 수정 후 승인`, `next = ('send',)`입니다. human_review까지 끝난 결과가 파일에 남았다는 뜻입니다.
3. 다시 켠 프로그램에서 새 그래프 객체가 같은 `next = ('send',)`와 초안을 읽습니다. `invoke(None, …)` 뒤에 `[발송]` 줄 하나만 출력되고 초안은 다시 만들어지지 않습니다. 모델 호출이 없었다는 뜻입니다. 대화 기록은 메시지 네 개입니다.
4. 실행 3에서 `[manage] 오래된 메시지 2개를 요약으로 바꿉니다` 줄이 출력되고, manage 뒤의 대화 기록은 최근 메시지 세 개와 요약 `SystemMessage` 하나입니다. 첫 메일과 첫 답장의 원문은 사라지고 요약 문장이 그 내용을 대신합니다. 발송 뒤의 최종 대화 기록 길이는 5입니다.